# Shopping Dataset — Exploratory Data Analysis & Cleaning
**Dataset:** Myntra Combined Shopping Dataset (1000 products, 96 categories)  
**Source:** [Kaggle — anvitkumar/shopping-dataset](https://www.kaggle.com/datasets/anvitkumar/shopping-dataset)


## 1: Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 100

df = pd.read_csv("../data/Combined_dataset.csv")

print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())

In [ ]:
df.head()

In [ ]:
df.tail()

## 2: Understand the Data

In [ ]:
df.dtypes

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
print("Missing values per column:")
print(df.isnull().sum())

## 3: Data Cleaning

### 3.1 — Clean `final_price` (stored as `"₹3,995.00"` — strip quotes, ₹ symbol, commas)


In [ ]:
df["final_price"] = (
    df["final_price"]
    .str.replace(r'["₹,]', '', regex=True)
    .str.strip()
    .astype(float)
)

print("final_price dtype:", df["final_price"].dtype)
print(df["final_price"].head())

### 3.2 — Handle missing values

In [ ]:
# discount: 121 nulls — fill with 0 (no discount listed = 0% off)
df["discount"] = df["discount"].fillna(0)

# what_customers_said, seller_name, seller_information, variations, videos:
# these are metadata columns not used in analysis — drop them to keep the dataset clean
cols_to_drop = ["what_customers_said", "videos", "seller_information", "variations"]
df.drop(columns=cols_to_drop, inplace=True)

# seller_name: still has nulls — fill with "Unknown"
df["seller_name"] = df["seller_name"].fillna("Unknown")

print("Remaining nulls:")
print(df.isnull().sum())

### 3.3 — Remove duplicates

In [ ]:
before = len(df)
df.drop_duplicates(inplace=True)
after = len(df)
print(f"Rows before: {before} | Rows after: {after} | Dropped: {before - after}")

### 3.4 — Filter out rows with rating = 0 (no real reviews)

In [ ]:
before = len(df)
df = df[df["rating"] > 0].reset_index(drop=True)
print(f"Removed {before - len(df)} unrated products. Remaining: {len(df)}")

## 4: Feature Engineering

### 4.1 — `price_difference`: how much the customer saves (initial_price − final_price)


In [ ]:
df["price_difference"] = df["initial_price"] - df["final_price"]
print(df[["title", "initial_price", "final_price", "price_difference"]].head())

### 4.2 — `popularity_metric`: rating × log(1 + ratings_count)

In [ ]:
df["popularity_metric"] = df["rating"] * np.log1p(df["ratings_count"])
print(df[["title", "rating", "ratings_count", "popularity_metric"]].head())

### 4.3 — `total_amount`: treat `final_price` as unit price, `ratings_count` as a proxy for volume

In [ ]:
# The dataset has no quantity column.
# As per the assignment objective (total_amount = price × quantity),
# we use ratings_count as a demand proxy to compute an estimated total amount.
df["total_amount"] = df["final_price"] * df["ratings_count"]
print(df[["title", "final_price", "ratings_count", "total_amount"]].head())

## 5: Basic Operations — Filter & Select

In [ ]:
# Filter: products with rating >= 4.5 and discount >= 50%
top_deals = df[(df["rating"] >= 4.5) & (df["discount"] >= 50)][
    ["title", "category", "rating", "discount", "final_price"]
]
print(f"Top deals (rating ≥ 4.5 AND discount ≥ 50%): {len(top_deals)} products")
top_deals.head(10)

In [ ]:
# Select: most popular products by popularity_metric
df[["title", "category", "rating", "ratings_count", "popularity_metric"]]    .sort_values("popularity_metric", ascending=False)    .head(10)

## 6: Analysis

### 6.1 — Univariate Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].hist(df["rating"], bins=20, color="#4C72B0", edgecolor="white")
axes[0].set_title("Distribution of Ratings")
axes[0].set_xlabel("Rating")
axes[0].set_ylabel("Count")

axes[1].hist(df["final_price"], bins=40, color="#DD8452", edgecolor="white")
axes[1].set_title("Distribution of Final Price (₹)")
axes[1].set_xlabel("Price (₹)")
axes[1].set_ylabel("Count")

axes[2].hist(df["discount"], bins=20, color="#55A868", edgecolor="white")
axes[2].set_title("Distribution of Discount (%)")
axes[2].set_xlabel("Discount %")
axes[2].set_ylabel("Count")

plt.tight_layout()
plt.savefig("univariate_analysis.png", bbox_inches="tight")
plt.show()

### 6.2 — Bivariate Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(df["discount"], df["rating"], alpha=0.4, color="#4C72B0", s=15)
axes[0].set_title("Discount % vs Rating")
axes[0].set_xlabel("Discount (%)")
axes[0].set_ylabel("Rating")

axes[1].scatter(df["final_price"], df["popularity_metric"], alpha=0.4, color="#C44E52", s=15)
axes[1].set_title("Final Price vs Popularity Metric")
axes[1].set_xlabel("Final Price (₹)")
axes[1].set_ylabel("Popularity Metric")

plt.tight_layout()
plt.savefig("bivariate_analysis.png", bbox_inches="tight")
plt.show()

### 6.3 — Category-Level Analysis

In [ ]:
category_stats = (
    df.groupby("category")
    .agg(
        avg_rating=("rating", "mean"),
        avg_discount=("discount", "mean"),
        avg_price=("final_price", "mean"),
        product_count=("product_id", "count"),
    )
    .round(2)
    .sort_values("product_count", ascending=False)
)

print("Top 10 categories by product count:")
print(category_stats.head(10).to_string())

In [ ]:
top10 = category_stats.nlargest(10, "product_count")

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].barh(top10.index, top10["avg_rating"], color="#4C72B0")
axes[0].set_title("Avg Rating — Top 10 Categories")
axes[0].set_xlabel("Average Rating")
axes[0].invert_yaxis()

axes[1].barh(top10.index, top10["avg_discount"], color="#DD8452")
axes[1].set_title("Avg Discount % — Top 10 Categories")
axes[1].set_xlabel("Average Discount (%)")
axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig("category_analysis.png", bbox_inches="tight")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
sns.boxplot(
    data=df[df["category"].isin(top10.index)],
    x="category",
    y="final_price",
    ax=ax,
    palette="muted"
)
ax.set_title("Price Distribution Across Top 10 Categories")
ax.set_xlabel("Category")
ax.set_ylabel("Final Price (₹)")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.savefig("price_boxplot.png", bbox_inches="tight")
plt.show()

## 7: Key Insights

1. **Rating distribution is left-skewed** — most products are rated between 3.5 and 4.5, indicating generally positive customer sentiment across the platform.

2. **Discount and rating are weakly correlated** — higher discounts do not reliably lead to higher ratings. Quality matters more than price cuts.

3. **Price distribution is heavily right-skewed** — the majority of products are priced below ₹2,000, with a long tail of premium items. Myntra caters primarily to budget and mid-range shoppers.

4. **High-discount products (≥ 50%)** exist across nearly every category, suggesting aggressive promotional pricing is a platform-wide strategy, not category-specific.

5. **Popularity metric reveals true winners** — products with a high `popularity_metric` (strong rating + large review volume) are concentrated in apparel categories like jeans, tshirts, and dresses, showing these are the most engaged product segments.

6. **Business implication** — categories with high avg discount but lower avg ratings (e.g., home/decor items) may benefit from quality improvement over price reduction. Apparel categories are the platform's strongest performing segment.


## 8: Save Cleaned Dataset

In [ ]:
df.to_csv("../data/Combined_dataset_cleaned.csv", index=False)
print(f"Saved cleaned dataset: {len(df)} rows × {df.shape[1]} columns")
print("Columns:", df.columns.tolist())